# 14 — Dual Paper Trading: Baseline vs ML Challenger

Bu notebook iki ayrı sanal portföyü paralel takip eder:

- **Baseline Robot**
- **ML Challenger:** `Big_Winner_Label_2R + LogisticRegression + Quantile_Q40`

Her iki portföy:

- 500.000 TL ile başlar.
- Aynı final risk ve çıkış kurallarını kullanır.
- Ayrı nakit ve pozisyon state dosyalarına sahiptir.
- Aynı piyasa gerçekleşme fiyatlarıyla kaydedilir.
- Günlük olarak ayrı alış/satış planları üretir.

ML challenger, Baseline Robot'un yerine geçmez. Gerçek yeni out-of-sample
karar yalnızca bu paralel paper-trading kaydıyla verilecektir.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.config import DataConfig
from src.data_loader import (
    load_bist_tickers,
    download_robot_bundle,
)
from src.data_quality import run_quality_pipeline
from src.features import add_indicators
from src.signals import (
    build_market_regime,
    add_robot_scores,
)
from src.presets import (
    FINAL_STRATEGY_CONFIG,
    FINAL_PORTFOLIO_CONFIG,
)
from src.ml_dataset import add_meta_features
from src.paper_trading import (
    load_paper_state,
    positions_dataframe,
)
from src.dual_paper import (
    load_challenger_deployment,
    choose_model_ready_signal_date,
    model_score_diagnostic,
    assert_model_scores_available,
    deployment_summary,
    build_challenger_prices,
    create_dual_daily_plan,
    compare_states,
    append_dual_equity_snapshot,
    save_dual_plans,
    record_buy_from_plan,
    record_sell_from_plan,
    persist_portfolio_state,
)


## 1. Kilitli challenger kararını ve modeli yükle


In [ ]:
challenger_deployment, challenger_model = (
    load_challenger_deployment(
        PROJECT_ROOT
    )
)

display(
    deployment_summary(
        challenger_deployment
    )
)


## 2. Güncel BIST100 ve XU100 verisini indir


In [ ]:
DOWNLOAD_START = (
    pd.Timestamp.today().normalize()
    - pd.Timedelta(days=900)
).strftime("%Y-%m-%d")

data_config = DataConfig(
    start=DOWNLOAD_START,
    end=None,
    auto_adjust=True,
    yfinance_repair=False,
)

tickers = load_bist_tickers(
    PROJECT_ROOT
    / "data"
    / "raw"
    / "bist100_sirketler.xlsx"
)

raw_stocks, raw_market, download_errors = (
    download_robot_bundle(
        tickers=tickers,
        config=data_config,
    )
)

print("Ticker sayısı:", len(tickers))
print("Hisse satırı:", len(raw_stocks))
print("Endeks satırı:", len(raw_market))
print("İndirme hatası:", len(download_errors))

if not download_errors.empty:
    display(download_errors)


## 3. Kalite kontrolü, Robot skorları ve ML özellikleri


In [ ]:
stock_quality = run_quality_pipeline(
    raw_stocks,
    config=data_config,
    apply_split_repairs=True,
)

market_quality = run_quality_pipeline(
    raw_market,
    config=data_config,
    apply_split_repairs=True,
)

stock_features = add_indicators(
    stock_quality.clean
)
market_features = add_indicators(
    market_quality.clean
)
market_regime = build_market_regime(
    market_features
)

baseline_prices = add_robot_scores(
    stock_features=stock_features,
    market_regime=market_regime,
    config=FINAL_STRATEGY_CONFIG,
    include_reasons=True,
)

featured_prices = add_meta_features(
    scored_prices=baseline_prices,
    market_features=market_features,
)

latest_stock_date = pd.Timestamp(
    baseline_prices["Date"].max()
)

model_ready_signal_date = (
    choose_model_ready_signal_date(
        featured_prices=featured_prices,
        minimum_coverage_ratio=0.60,
    )
)

prediction_start = (
    model_ready_signal_date
    - pd.Timedelta(days=90)
)

challenger_prices = build_challenger_prices(
    featured_prices=featured_prices,
    fitted_model=challenger_model,
    deployment=challenger_deployment,
    prediction_start=prediction_start,
    prediction_end=model_ready_signal_date,
)

score_diagnostic = model_score_diagnostic(
    baseline_prices=baseline_prices,
    challenger_prices=challenger_prices,
    signal_date=model_ready_signal_date,
    deployment=challenger_deployment,
)

assert_model_scores_available(
    score_diagnostic
)

print("En son hisse fiyat tarihi:", latest_stock_date)
print(
    "Model-ready ortak sinyal tarihi:",
    model_ready_signal_date,
)
print(
    "Veri gecikmesi (takvim günü):",
    (latest_stock_date - model_ready_signal_date).days,
)

print(
    "Model-ready tarihte Baseline AL:",
    baseline_prices.loc[
        baseline_prices["Date"].eq(
            model_ready_signal_date
        ),
        "Signal",
    ].eq("AL").sum(),
)

print(
    "Model-ready tarihte Challenger AL:",
    challenger_prices.loc[
        challenger_prices["Date"].eq(
            model_ready_signal_date
        ),
        "Signal",
    ].eq("AL").sum(),
)

display(score_diagnostic)


## 4. İki ayrı paper-trading state dosyasını yükle


In [ ]:
DUAL_ROOT = (
    PROJECT_ROOT
    / "results"
    / "paper_trading"
    / "dual"
)

BASELINE_STATE_PATH = (
    DUAL_ROOT
    / "baseline"
    / "state.json"
)
BASELINE_TRADES_PATH = (
    DUAL_ROOT
    / "baseline"
    / "closed_trades.csv"
)

CHALLENGER_STATE_PATH = (
    DUAL_ROOT
    / "challenger"
    / "state.json"
)
CHALLENGER_TRADES_PATH = (
    DUAL_ROOT
    / "challenger"
    / "closed_trades.csv"
)

DUAL_HISTORY_PATH = (
    DUAL_ROOT
    / "dual_equity_history.csv"
)
DUAL_PLANS_DIR = (
    DUAL_ROOT
    / "daily_plans"
)

baseline_state = load_paper_state(
    path=BASELINE_STATE_PATH,
    initial_capital=(
        FINAL_PORTFOLIO_CONFIG.initial_capital
    ),
)

challenger_state = load_paper_state(
    path=CHALLENGER_STATE_PATH,
    initial_capital=(
        FINAL_PORTFOLIO_CONFIG.initial_capital
    ),
)

print("BASELINE POZİSYONLARI")
display(positions_dataframe(baseline_state))

print("CHALLENGER POZİSYONLARI")
display(positions_dataframe(challenger_state))


## 5. İki portföy için eş zamanlı günlük plan üret


In [ ]:
dual_plan = create_dual_daily_plan(
    baseline_prices=baseline_prices,
    challenger_prices=challenger_prices,
    baseline_state=baseline_state,
    challenger_state=challenger_state,
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    signal_date=model_ready_signal_date,
    minimum_coverage_ratio=0.60,
)

display(dual_plan.summary_comparison)

print("ALIŞ KARŞILAŞTIRMASI")
display(
    dual_plan.buy_comparison.sort_values(
        ["Ticker", "Portfolio"]
    )
)

print("SATIŞ KARŞILAŞTIRMASI")
display(
    dual_plan.sell_comparison.sort_values(
        ["Ticker", "Portfolio"]
    )
)


`Appears_In_Both=True` olan hisseler iki portföyün ortak sinyalidir.
Yalnızca Baseline'da bulunanlar ML tarafından elenen sinyallerdir.


In [ ]:
plan_paths = save_dual_plans(
    dual_plan=dual_plan,
    output_root=DUAL_PLANS_DIR,
)

for name, path in plan_paths.items():
    print(name, "→", path)


## 6. İki portföyü güncel kapanışlarla değerle


In [ ]:
latest_rows = (
    baseline_prices.loc[
        baseline_prices["Date"].eq(
            dual_plan.signal_date
        )
    ]
    .drop_duplicates("Ticker", keep="last")
)

latest_price_map = dict(
    zip(
        latest_rows["Ticker"],
        latest_rows["Close"],
    )
)

dual_equity = compare_states(
    baseline_state=baseline_state,
    challenger_state=challenger_state,
    latest_prices=latest_price_map,
)

display(dual_equity)

append_dual_equity_snapshot(
    comparison=dual_equity,
    signal_date=dual_plan.signal_date,
    path=DUAL_HISTORY_PATH,
)


## 7. Gerçekleşen alışları kaydet

Aşağıdaki örnekler yorum satırındadır. Emir gerçekten dolduktan sonra ilgili
portföyün planını ve state dosyasını kullan.

Aynı hisse iki portföyde de varsa aynı piyasa `FILL_PRICE` değerini kullan;
lot sayıları her portföyün kendi nakit ve risk durumuna göre hesaplanır.


In [ ]:
# BASELINE ALIŞ ÖRNEĞİ
#
# TICKER = "AKBNK.IS"
# FILL_DATE = "2026-07-27"
# FILL_PRICE = 70.25
#
# baseline_equity_now = float(
#     dual_equity.loc[
#         dual_equity["Portfolio"].eq("Baseline_Robot"),
#         "Equity",
#     ].iloc[0]
# )
#
# baseline_state, baseline_fill = record_buy_from_plan(
#     state=baseline_state,
#     buy_orders=dual_plan.baseline.buy_orders,
#     ticker=TICKER,
#     fill_date=FILL_DATE,
#     fill_price=FILL_PRICE,
#     strategy_config=FINAL_STRATEGY_CONFIG,
#     portfolio_config=FINAL_PORTFOLIO_CONFIG,
#     current_equity=baseline_equity_now,
# )
#
# persist_portfolio_state(
#     baseline_state,
#     BASELINE_STATE_PATH,
# )
#
# display(pd.DataFrame([baseline_fill]))


In [ ]:
# CHALLENGER ALIŞ ÖRNEĞİ
#
# TICKER = "AKBNK.IS"
# FILL_DATE = "2026-07-27"
# FILL_PRICE = 70.25
#
# challenger_equity_now = float(
#     dual_equity.loc[
#         dual_equity["Portfolio"].eq("ML_Challenger"),
#         "Equity",
#     ].iloc[0]
# )
#
# challenger_state, challenger_fill = record_buy_from_plan(
#     state=challenger_state,
#     buy_orders=dual_plan.challenger.buy_orders,
#     ticker=TICKER,
#     fill_date=FILL_DATE,
#     fill_price=FILL_PRICE,
#     strategy_config=FINAL_STRATEGY_CONFIG,
#     portfolio_config=FINAL_PORTFOLIO_CONFIG,
#     current_equity=challenger_equity_now,
# )
#
# persist_portfolio_state(
#     challenger_state,
#     CHALLENGER_STATE_PATH,
# )
#
# display(pd.DataFrame([challenger_fill]))


## 8. Gerçekleşen satışları kaydet


In [ ]:
# BASELINE SATIŞ ÖRNEĞİ
#
# TICKER = "AKBNK.IS"
# FILL_DATE = "2026-08-10"
# FILL_PRICE = 76.80
#
# baseline_state, baseline_trade = record_sell_from_plan(
#     state=baseline_state,
#     sell_orders=dual_plan.baseline.sell_orders,
#     ticker=TICKER,
#     fill_date=FILL_DATE,
#     fill_price=FILL_PRICE,
#     portfolio_config=FINAL_PORTFOLIO_CONFIG,
#     trades_path=BASELINE_TRADES_PATH,
# )
#
# persist_portfolio_state(
#     baseline_state,
#     BASELINE_STATE_PATH,
# )
#
# display(pd.DataFrame([baseline_trade]))


In [ ]:
# CHALLENGER SATIŞ ÖRNEĞİ
#
# TICKER = "AKBNK.IS"
# FILL_DATE = "2026-08-10"
# FILL_PRICE = 76.80
#
# challenger_state, challenger_trade = record_sell_from_plan(
#     state=challenger_state,
#     sell_orders=dual_plan.challenger.sell_orders,
#     ticker=TICKER,
#     fill_date=FILL_DATE,
#     fill_price=FILL_PRICE,
#     portfolio_config=FINAL_PORTFOLIO_CONFIG,
#     trades_path=CHALLENGER_TRADES_PATH,
# )
#
# persist_portfolio_state(
#     challenger_state,
#     CHALLENGER_STATE_PATH,
# )
#
# display(pd.DataFrame([challenger_trade]))


## Günlük çalışma düzeni

Piyasa kapandıktan sonra:

1. Bölüm 1–6'yı çalıştır.
2. Baseline ve Challenger alış/satış farklarını incele.
3. Ertesi gün gerçekleşen emirleri Bölüm 7–8 ile kaydet.
4. İki portföyde aynı emir varsa aynı gerçekleşme fiyatını kullan.
5. ML modelini, threshold'u veya strateji parametrelerini paper-trading
   değerlendirmesi tamamlanmadan değiştirme.

İlk değerlendirme için asgari hedef:

- En az 50 kapalı challenger işlemi
- Tercihen 3–6 aylık piyasa dönemi
- Yükseliş, yatay ve düzeltme günlerinin birlikte görülmesi
